In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "raw"

history = pd.read_csv(DATA_DIR / "history.csv")
workload = pd.read_csv(DATA_DIR / "Cell_c.csv")

print("Carbon dataset:", history.shape)
print("Workload dataset:", workload.shape)

print("\nCarbon columns:")
print(history.columns.tolist())

print("\nWorkload columns:")
print(workload.columns.tolist())

Carbon dataset: (38976, 5)
Workload dataset: (7200798, 5)

Carbon columns:
['provider', 'region', 'timestamp', 'carbon_intensity_gco2_kwh', 'renewable_percentage']

Workload columns:
['Job_ID', 'Task_ID', 'Arrival_Time', 'CPU', 'Memory']


In [5]:
workload = pd.read_csv(
    BASE_DIR / "data" / "preprocessed" / "workload.csv"
)

print("Workload shape:", workload.shape)
print(workload.columns.tolist())
display(workload.head())

Workload shape: (7200798, 5)
['Job_ID', 'Task_ID', 'Arrival_Time', 'CPU', 'Memory']


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185


In [6]:
print(workload.columns.tolist())
print(workload.dtypes)
display(workload.head(3).T)

['Job_ID', 'Task_ID', 'Arrival_Time', 'CPU', 'Memory']
Job_ID            int64
Task_ID           int64
Arrival_Time    float64
CPU             float64
Memory          float64
dtype: object


,0,1,2
Job_ID,7.369581e+11,7.369581e+11,7.369581e+11
Task_ID,0.000000e+00,0.000000e+00,0.000000e+00
Arrival_Time,6.012637e+05,6.013400e+05,6.019755e+05
CPU,1.798916e-02,8.105278e-03,1.124887e-01
Memory,5.722046e-05,5.722046e-05,2.374649e-04


In [7]:
region_config = pd.read_csv(
    BASE_DIR / "data" / "preprocessed" / "region_config.csv"
)

print("Shape:", region_config.shape)
print(region_config.columns.tolist())
display(region_config.head().T)

Shape: (1160, 13)
['datacenter_id', 'provider', 'region', 'host_id', 'pes', 'mips_per_pe', 'ram_mb', 'storage_mb', 'bandwidth_mbps', 'idle_power_w', 'max_power_w', 'pue', 'cost_per_hour']


,0,1,2,3,4
datacenter_id,DC001,DC001,DC001,DC001,DC001
provider,aws,aws,aws,aws,aws
region,af-south-1,af-south-1,af-south-1,af-south-1,af-south-1
host_id,H01,H02,H03,H04,H05
pes,4,8,8,12,16
mips_per_pe,800,1000,1200,1500,2000
ram_mb,8192,16384,16384,24576,32768
storage_mb,500000,1000000,1000000,1500000,2000000
bandwidth_mbps,5000,10000,10000,15000,20000
idle_power_w,130,160,162,192,222


In [8]:
print("Shape:", history.shape)
print(history.columns.tolist())
display(history.head().T)

Shape: (38976, 5)
['provider', 'region', 'timestamp', 'carbon_intensity_gco2_kwh', 'renewable_percentage']


,0,1,2,3,4
provider,aws,aws,aws,aws,aws
region,af-south-1,af-south-1,af-south-1,af-south-1,af-south-1
timestamp,2026-08-19T00:53:00.211592+00:00,2026-08-19T02:17:53.174894+00:00,2026-08-19T03:42:26.812833+00:00,2026-08-19T04:34:02.547742+00:00,2026-08-19T05:25:46.493498+00:00
carbon_intensity_gco2_kwh,795.6,795.6,780.0,780.0,780.0
renewable_percentage,5.0,5.0,8.0,8.0,8.0


In [9]:
print("WORKLOAD COLUMNS:")
print(workload.columns.tolist())

print("\nRESOURCE CONFIG COLUMNS:")
print(region_config.columns.tolist())

print("\nCARBON COLUMNS:")
print(history.columns.tolist())

WORKLOAD COLUMNS:
['Job_ID', 'Task_ID', 'Arrival_Time', 'CPU', 'Memory']

RESOURCE CONFIG COLUMNS:
['datacenter_id', 'provider', 'region', 'host_id', 'pes', 'mips_per_pe', 'ram_mb', 'storage_mb', 'bandwidth_mbps', 'idle_power_w', 'max_power_w', 'pue', 'cost_per_hour']

CARBON COLUMNS:
['provider', 'region', 'timestamp', 'carbon_intensity_gco2_kwh', 'renewable_percentage']


In [10]:
history["timestamp"] = pd.to_datetime(history["timestamp"], errors="coerce")

history["carbon_intensity_gco2_kwh"] = pd.to_numeric(
    history["carbon_intensity_gco2_kwh"], errors="coerce"
)

history["renewable_percentage"] = pd.to_numeric(
    history["renewable_percentage"], errors="coerce"
)

history = history.dropna(
    subset=[
        "provider",
        "region",
        "timestamp",
        "carbon_intensity_gco2_kwh",
        "renewable_percentage"
    ]
)

history = history.drop_duplicates()

history = history.sort_values(
    ["provider", "region", "timestamp"]
).reset_index(drop=True)

history["hour"] = history["timestamp"].dt.hour
history["day_of_week"] = history["timestamp"].dt.dayofweek
history["month"] = history["timestamp"].dt.month

print("Carbon dataset:", history.shape)
display(history.head())

Carbon dataset: (38976, 8)


,provider,region,timestamp,carbon_intensity_gco2_kwh,renewable_percentage,hour,day_of_week,month
0,aws,af-south-1,2026-08-19 00:53:00.211592+00:00,795.6,5.0,0,2,8
1,aws,af-south-1,2026-08-19 02:17:53.174894+00:00,795.6,5.0,2,2,8
2,aws,af-south-1,2026-08-19 03:42:26.812833+00:00,780.0,8.0,3,2,8
3,aws,af-south-1,2026-08-19 04:34:02.547742+00:00,780.0,8.0,4,2,8
4,aws,af-south-1,2026-08-19 05:25:46.493498+00:00,780.0,8.0,5,2,8


In [11]:
numeric_cols = [
    "pes",
    "mips_per_pe",
    "ram_mb",
    "storage_mb",
    "bandwidth_mbps",
    "idle_power_w",
    "max_power_w",
    "pue",
    "cost_per_hour"
]

for col in numeric_cols:
    region_config[col] = pd.to_numeric(region_config[col], errors="coerce")

region_config = region_config.dropna(
    subset=["datacenter_id", "provider", "region", "host_id"] + numeric_cols
)

region_config = region_config.drop_duplicates()

region_config["total_cpu_mips"] = (
    region_config["pes"] * region_config["mips_per_pe"]
)

print("Resource configuration:", region_config.shape)
display(region_config.head())

Resource configuration: (1160, 14)


,datacenter_id,provider,region,host_id,pes,mips_per_pe,ram_mb,storage_mb,bandwidth_mbps,idle_power_w,max_power_w,pue,cost_per_hour,total_cpu_mips
0,DC001,aws,af-south-1,H01,4,800,8192,500000,5000,130,262,1.2,0.085,3200
1,DC001,aws,af-south-1,H02,8,1000,16384,1000000,10000,160,324,1.2,0.090,8000
2,DC001,aws,af-south-1,H03,8,1200,16384,1000000,10000,162,326,1.2,0.095,9600
3,DC001,aws,af-south-1,H04,12,1500,24576,1500000,15000,192,388,1.2,0.100,18000
4,DC001,aws,af-south-1,H05,16,2000,32768,2000000,20000,222,450,1.2,0.105,32000


In [12]:
for col in ["Arrival_Time", "CPU", "Memory"]:
    workload[col] = pd.to_numeric(workload[col], errors="coerce")

workload = workload.dropna(
    subset=["Job_ID", "Task_ID", "Arrival_Time", "CPU", "Memory"]
)

workload = workload[
    (workload["Arrival_Time"] >= 0)
    & (workload["CPU"] > 0)
    & (workload["Memory"] > 0)
]

print("Clean workload shape:", workload.shape)
display(workload.head())

Clean workload shape: (7010740, 5)


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185


In [ ]:
from pathlib import Path

OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

history.to_csv(
    OUTPUT_DIR / "carbon_intensity.csv",
    index=False
)

region_config.to_csv(
    OUTPUT_DIR / "region_config.csv",
    index=False
)

workload.to_csv(
    OUTPUT_DIR / "workload.csv",
    index=False
)

print("All processed datasets saved successfully!")

All processed datasets saved successfully!


In [3]:
import sys
import importlib
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent
sys.path.insert(0, str(BASE_DIR / "src" / "cloudsim"))

import preprocessing
importlib.reload(preprocessing)

print(type(preprocessing.pd))
print(hasattr(preprocessing.pd, "read_csv"))

<class 'module'>
True


In [4]:
history = preprocessing.load_carbon_data(
    BASE_DIR / "data" / "raw" / "history.csv"
)

print("Carbon data shape:", history.shape)
print(history.head())

Carbon data shape: (38976, 8)
  provider      region                        timestamp  \
0      aws  af-south-1 2026-08-19 00:53:00.211592+00:00   
1      aws  af-south-1 2026-08-19 02:17:53.174894+00:00   
2      aws  af-south-1 2026-08-19 03:42:26.812833+00:00   
3      aws  af-south-1 2026-08-19 04:34:02.547742+00:00   
4      aws  af-south-1 2026-08-19 05:25:46.493498+00:00   

   carbon_intensity_gco2_kwh  renewable_percentage  hour  day_of_week  month  
0                      795.6                   5.0     0            2      8  
1                      795.6                   5.0     2            2      8  
2                      780.0                   8.0     3            2      8  
3                      780.0                   8.0     4            2      8  
4                      780.0                   8.0     5            2      8  


In [5]:
resources = preprocessing.load_resource_data(
    BASE_DIR / "data" / "preprocessed" / "region_config.csv"
)

print("Resource data shape:", resources.shape)
print(resources.head())

Resource data shape: (1160, 14)
  datacenter_id provider      region host_id  pes  mips_per_pe  ram_mb  \
0         DC001      aws  af-south-1     H01    4          800    8192   
1         DC001      aws  af-south-1     H02    8         1000   16384   
2         DC001      aws  af-south-1     H03    8         1200   16384   
3         DC001      aws  af-south-1     H04   12         1500   24576   
4         DC001      aws  af-south-1     H05   16         2000   32768   

   storage_mb  bandwidth_mbps  idle_power_w  max_power_w  pue  cost_per_hour  \
0      500000            5000           130          262  1.2          0.085   
1     1000000           10000           160          324  1.2          0.090   
2     1000000           10000           162          326  1.2          0.095   
3     1500000           15000           192          388  1.2          0.100   
4     2000000           20000           222          450  1.2          0.105   

   total_cpu_mips  
0            3200  
1 

In [6]:
workload = preprocessing.load_workload_data(
    BASE_DIR / "data" / "raw" / "Cell_c.csv"
)

print("Workload data shape:", workload.shape)
print(workload.head())

Workload data shape: (7010740, 5)
         Job_ID  Task_ID  Arrival_Time       CPU    Memory
0  736958118313        0    601263.722  0.017989  0.000057
1  736958118321        0    601339.991  0.008105  0.000057
2  736958118528        0    601975.491  0.112489  0.000237
3  736958125656        0    604299.116  0.055855  0.001207
4  736959135511        0    608551.032  0.323642  0.003185


In [7]:
preprocessing.save_processed_data(
    history,
    BASE_DIR / "data" / "processed" / "carbon_intensity.csv"
)

preprocessing.save_processed_data(
    resources,
    BASE_DIR / "data" / "processed" / "region_config.csv"
)

preprocessing.save_processed_data(
    workload,
    BASE_DIR / "data" / "processed" / "workload.csv"
)

Saved 38976 rows to c:\Users\LENOVO\Downloads\AI-Carbon-Aware-Cloud-Scheduler\AI-Carbon-Aware-Cloud-Scheduler\data\processed\carbon_intensity.csv
Saved 1160 rows to c:\Users\LENOVO\Downloads\AI-Carbon-Aware-Cloud-Scheduler\AI-Carbon-Aware-Cloud-Scheduler\data\processed\region_config.csv
Saved 7010740 rows to c:\Users\LENOVO\Downloads\AI-Carbon-Aware-Cloud-Scheduler\AI-Carbon-Aware-Cloud-Scheduler\data\processed\workload.csv
